# **Notebook 5: Solution V1 — RAG Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `./chroma_db/` — Created by Notebook 4
- [ ] `df_test.csv` — Created by Notebook 2
- [ ] `outputs.json` — Created by Notebooks 3+4
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `v1_metrics.csv` — Per-row Baseline vs V1 scores _(Evidence for comparative analysis)_

---

In [1]:
# ============================================================
# COLAB PROJECT SETUP
# ============================================================

from google.colab import drive
from pathlib import Path
import os

# Mount Google Drive
drive.mount("/content/drive")

# Permanent project folder in Google Drive
DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/Hybrid_RAG_Customer_Support"
)

# Temporary workspace for the current Colab runtime
LOCAL_PROJECT_DIR = Path(
    "/content/Hybrid_RAG_Customer_Support"
)

LOCAL_PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Work from the temporary Colab directory
os.chdir(LOCAL_PROJECT_DIR)

print("Google Drive project:", DRIVE_PROJECT_DIR)
print("Local Colab workspace:", LOCAL_PROJECT_DIR)
print("Current working directory:", Path.cwd())

Mounted at /content/drive
Google Drive project: /content/drive/MyDrive/Hybrid_RAG_Customer_Support
Local Colab workspace: /content/Hybrid_RAG_Customer_Support
Current working directory: /content/Hybrid_RAG_Customer_Support


In [2]:
# ============================================================
# VERIFY GPU RUNTIME
# ============================================================

import torch

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU runtime is not enabled. In Colab, go to "
        "Runtime → Change runtime type → T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = (
    torch.cuda.get_device_properties(0).total_memory
    / 1024**3
)

print("GPU detected    :", gpu_name)
print(f"GPU memory      : {gpu_memory_gb:.2f} GB")
print("CUDA version    :", torch.version.cuda)
print("\nGPU runtime is enabled successfully.")

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU detected    : Tesla T4
GPU memory      : 14.56 GB
CUDA version    : 12.8

GPU runtime is enabled successfully.


### **Task 3.3: Evaluate Solution V1**

> This task is split into five measurements (3.3.1–3.3.5). Run the shared setup cell below first (it loads the model, ChromaDB, and test data), then work through each measurement.

**── Shared setup ──**
Load the base model, reload ChromaDB (same embedding model as NB4), and load `df_test.csv` + `outputs.json`. Define helper functions `generate_baseline()` and `generate_naive_rag()` here so every subtask below can reuse them.

In [3]:
# YOUR CODE HERE
constraints = """
requests==2.32.4
fsspec==2025.3.0
numpy>=1.26,<2.1
websockets>=15.0.1,<16
protobuf>=5.26.1,<6
rich>=10.11,<14
tokenizers==0.21.4
opentelemetry-api==1.42.1
opentelemetry-sdk==1.42.1
opentelemetry-semantic-conventions==0.63b1
opentelemetry-exporter-otlp-proto-grpc==1.42.1
opentelemetry-exporter-otlp-proto-common==1.42.1
opentelemetry-proto==1.42.1
"""

with open("/tmp/colab_constraints.txt", "w") as file:
    file.write(constraints)

%pip install -q \
    --constraint /tmp/colab_constraints.txt \
    "chromadb==1.0.20" \
    "sentence-transformers==3.4.1" \
    "transformers==4.53.2" \
    "tokenizers==0.21.4" \
    "bitsandbytes==0.46.1" \
    "accelerate==1.8.1" \
    "jedi>=0.16"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.9/275.9 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 113.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 112.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 128.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [4]:
import json
import random
from pathlib import Path

import chromadb
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

RANDOM_STATE = 42
MAX_NEW_TOKENS = 120

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)

if not torch.cuda.is_available():
    raise RuntimeError(
        "A T4 GPU runtime is required for Notebook 5."
    )

# Define required artefact paths
DF_TEST_PATH = (
    DRIVE_PROJECT_DIR
    / "artifacts"
    / "notebook_2"
    / "df_test.csv"
)

OUTPUTS_PATH = (
    DRIVE_PROJECT_DIR
    / "artifacts"
    / "notebook_3"
    / "outputs.json"
)

CHROMA_DB_PATH = (
    DRIVE_PROJECT_DIR
    / "artifacts"
    / "notebook_4"
    / "chroma_db"
)

for required_path in [
    DF_TEST_PATH,
    OUTPUTS_PATH,
    CHROMA_DB_PATH
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required artefact was not found:\n{required_path}"
        )

# Load held-out test data
df_test = pd.read_csv(DF_TEST_PATH)

REQUIRED_TEST_COLUMNS = {
    "instruction",
    "intent",
    "category"
}

missing_test_columns = (
    REQUIRED_TEST_COLUMNS
    - set(df_test.columns)
)

if missing_test_columns:
    raise ValueError(
        f"df_test.csv is missing columns: "
        f"{sorted(missing_test_columns)}"
    )

assert not df_test[
    list(REQUIRED_TEST_COLUMNS)
].isna().any().any()

print("Held-out test data loaded")
print(f"Test records : {len(df_test):,}")
print(f"Test intents : {df_test['intent'].nunique()}")

Held-out test data loaded
Test records : 400
Test intents : 27


In [5]:
# Load outputs from Notebooks 3 and 4
with open(
    OUTPUTS_PATH,
    "r",
    encoding="utf-8"
) as file:
    saved_outputs = json.load(file)

required_output_fields = {
    "test_query",
    "ground_truth",
    "baseline_output",
    "naive_rag_output",
    "naive_rag_retrieval"
}

missing_output_fields = (
    required_output_fields
    - set(saved_outputs)
)

if missing_output_fields:
    raise ValueError(
        f"outputs.json is missing fields: "
        f"{sorted(missing_output_fields)}"
    )

# Recover the fixed model and retrieval configuration
MODEL_ID = saved_outputs.get(
    "model_id",
    "Qwen/Qwen2.5-1.5B-Instruct"
)

retrieval_config = saved_outputs[
    "naive_rag_retrieval"
]

EMBEDDING_MODEL_ID = retrieval_config.get(
    "embedding_model_id",
    "sentence-transformers/all-MiniLM-L6-v2"
)

COLLECTION_NAME = retrieval_config.get(
    "collection_name",
    "corporate_policy_chunks"
)

DEFAULT_TOP_K = int(
    retrieval_config.get("top_k", 1)
)

print("\nRecovered configuration")
print("-" * 55)
print(f"Generation model : {MODEL_ID}")
print(f"Embedding model  : {EMBEDDING_MODEL_ID}")
print(f"Collection name  : {COLLECTION_NAME}")
print(f"Default top-k    : {DEFAULT_TOP_K}")


Recovered configuration
-------------------------------------------------------
Generation model : Qwen/Qwen2.5-1.5B-Instruct
Embedding model  : sentence-transformers/all-MiniLM-L6-v2
Collection name  : corporate_policy_chunks
Default top-k    : 1


In [6]:
# Reload the embedding model on CPU
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_ID,
    device="cpu"
)

sample_embedding = embedding_model.encode(
    ["embedding validation"],
    normalize_embeddings=True
)

assert sample_embedding.shape == (1, 384)

print("\nEmbedding model loaded successfully")
print(f"Embedding dimension: {sample_embedding.shape[1]}")

# Reload the persisted Chroma collection
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DB_PATH)
)

chroma_collection = chroma_client.get_collection(
    name=COLLECTION_NAME
)

indexed_chunk_count = chroma_collection.count()

if indexed_chunk_count == 0:
    raise RuntimeError(
        "The persisted Chroma collection contains no records."
    )

print("\nChroma collection loaded successfully")
print(f"Indexed chunks: {indexed_chunk_count}")

# Load Qwen in 4-bit
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

model.eval()

MODEL_DEVICE = next(model.parameters()).device

print("\nGeneration model loaded successfully")
print(f"GPU          : {torch.cuda.get_device_name(0)}")
print(f"Model device : {MODEL_DEVICE}")
print(
    f"Footprint    : "
    f"{model.get_memory_footprint() / 1024**3:.2f} GB"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Generation model loaded successfully
GPU          : Tesla T4
Model device : cuda:0
Footprint    : 1.05 GB


In [7]:
# Shared deterministic text-generation helper
def generate_from_messages(
    messages,
    max_new_tokens=MAX_NEW_TOKENS
):
    """Generate one deterministic assistant response."""

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False
    ).to(MODEL_DEVICE)

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_token_ids = generated_ids[
        0,
        model_inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        new_token_ids,
        skip_special_tokens=True
    ).strip()

# Baseline-generation helper
def generate_baseline(
    query,
    max_new_tokens=MAX_NEW_TOKENS
):
    """Generate an answer without SOP retrieval."""

    messages = [
        {
            "role": "system",
            "content": (
                "You are a customer-support assistant. "
                "Answer the customer's question clearly "
                "and concisely."
            )
        },
        {
            "role": "user",
            "content": str(query).strip()
        }
    ]

    return generate_from_messages(
        messages,
        max_new_tokens=max_new_tokens
    )

# Raw-query retrieval helper
def retrieve_naive_context(
    query,
    k=DEFAULT_TOP_K
):
    """Retrieve top-k SOP chunks using the raw customer query."""

    query_embedding = embedding_model.encode(
        [str(query).strip()],
        normalize_embeddings=True
    ).astype(np.float32)

    results = chroma_collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    if not documents:
        raise RuntimeError(
            "Chroma returned no documents."
        )

    retrieved_items = []

    for rank, (
        document,
        metadata,
        distance
    ) in enumerate(
        zip(documents, metadatas, distances),
        start=1
    ):
        retrieved_items.append({
            "rank": rank,
            "context": document,
            "source_file": metadata.get(
                "source_file",
                "unknown"
            ),
            "chunk_id": metadata.get(
                "chunk_id",
                "unknown"
            ),
            "distance": float(distance)
        })

    return retrieved_items

In [8]:
# Naive-RAG generation helper
def generate_naive_rag(
    query,
    k=DEFAULT_TOP_K,
    max_new_tokens=MAX_NEW_TOKENS,
    return_details=False
):
    """
    Retrieve using the raw query and generate an SOP-grounded
    response.
    """

    retrieved_items = retrieve_naive_context(
        query=query,
        k=k
    )

    context_blocks = []

    for item in retrieved_items:
        context_blocks.append(
            f"[Source: {item['source_file']} | "
            f"Chunk: {item['chunk_id']}]\n"
            f"{item['context']}"
        )

    combined_context = "\n\n".join(
        context_blocks
    )

    system_prompt = (
        "You are a customer-support assistant. "
        "Answer strictly using only the supplied SOP context. "
        "Do not invent timelines, procedures, departments, "
        "contact details, refunds, or guarantees. If the context "
        "does not contain enough information, state that the "
        "available policy does not provide enough information."
    )

    user_prompt = f"""
Customer query:
{str(query).strip()}

Retrieved SOP context:
{combined_context}

Provide a concise policy-grounded answer to the customer.
""".strip()

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    response = generate_from_messages(
        messages,
        max_new_tokens=max_new_tokens
    )

    result = {
        "query": str(query).strip(),
        "response": response,
        "retrieved_items": retrieved_items,
        "retrieved_source": (
            retrieved_items[0]["source_file"]
        ),
        "retrieved_chunk_id": (
            retrieved_items[0]["chunk_id"]
        ),
        "retrieved_context": combined_context
    }

    if return_details:
        return result

    return response

# Final setup validation
assert callable(generate_baseline)
assert callable(generate_naive_rag)
assert callable(retrieve_naive_context)

print("\nShared setup completed successfully")
print("-" * 55)
print("Available objects:")
print("  df_test")
print("  saved_outputs")
print("  generate_baseline(query)")
print("  retrieve_naive_context(query, k)")
print("  generate_naive_rag(query, k)")


Shared setup completed successfully
-------------------------------------------------------
Available objects:
  df_test
  saved_outputs
  generate_baseline(query)
  retrieve_naive_context(query, k)
  generate_naive_rag(query, k)


#### **3.3.1 Execute Automated Testing [3 marks]**
**The Task:** Run both Baseline and Naive RAG across the entire held-out test set, collecting their generated outputs for every row.

**Hints & Tips:**
* Loop over `df_test` rows; for each query call both `generate_baseline()` and `generate_naive_rag()`.
* Store raw outputs in a list of dicts so the later measurements can score them.
* This is the most time-consuming cell — if constrained, `df_test.sample(50)` is acceptable.

**Learner Inference:** Automated testing across the full set gives statistically meaningful results, not a single cherry-picked query.

In [9]:
# YOUR CODE HERE
import time
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

RANDOM_STATE = 42

# The notebook explicitly permits 50 records when runtime is constrained.
# Change to None to evaluate the complete 400-record test set.
EVAL_SAMPLE_SIZE = 50

if EVAL_SAMPLE_SIZE is None:
    evaluation_df = df_test.copy()
else:
    if EVAL_SAMPLE_SIZE > len(df_test):
        raise ValueError(
            f"Requested {EVAL_SAMPLE_SIZE} records, but the test set "
            f"contains only {len(df_test)}."
        )

    evaluation_df = (
        df_test
        .sample(
            n=EVAL_SAMPLE_SIZE,
            random_state=RANDOM_STATE
        )
        .reset_index(drop=True)
    )

print("Automated evaluation configuration")
print("-" * 55)
print(f"Complete held-out test set : {len(df_test):,}")
print(f"Records being evaluated    : {len(evaluation_df):,}")
print("Systems per record         : Baseline and Naive RAG")
print(
    f"Total generations          : "
    f"{len(evaluation_df) * 2:,}"
)

Automated evaluation configuration
-------------------------------------------------------
Complete held-out test set : 400
Records being evaluated    : 50
Systems per record         : Baseline and Naive RAG
Total generations          : 100


In [10]:
# Prepare a checkpoint path
NOTEBOOK_5_ARTIFACT_DIR = (
    DRIVE_PROJECT_DIR
    / "artifacts"
    / "notebook_5"
)

CHECKPOINT_PATH = (
    NOTEBOOK_5_ARTIFACT_DIR
    / "automated_test_checkpoint.csv"
)

NOTEBOOK_5_ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Run both systems across the evaluation set
automated_test_results = []
evaluation_start_time = time.perf_counter()

for row_number, row in tqdm(
    evaluation_df.iterrows(),
    total=len(evaluation_df),
    desc="Running Baseline and Naive RAG"
):
    query = str(row["instruction"]).strip()

    result = {
        "evaluation_id": row_number,
        "instruction": query,
        "true_intent": row["intent"],
        "true_category": row["category"],
        "baseline_output": None,
        "naive_rag_output": None,
        "retrieved_source": None,
        "retrieved_chunk_id": None,
        "retrieval_distance": None,
        "baseline_success": False,
        "naive_rag_success": False,
        "baseline_error": None,
        "naive_rag_error": None
    }

    # Baseline inference
    try:
        baseline_start = time.perf_counter()

        result["baseline_output"] = generate_baseline(
            query
        )

        result["baseline_latency_seconds"] = round(
            time.perf_counter() - baseline_start,
            3
        )

        result["baseline_success"] = bool(
            result["baseline_output"]
            and result["baseline_output"].strip()
        )

    except Exception as error:
        result["baseline_error"] = (
            f"{type(error).__name__}: {error}"
        )
        result["baseline_latency_seconds"] = None

    # Naive RAG inference
    try:
        rag_start = time.perf_counter()

        rag_result = generate_naive_rag(
            query=query,
            k=DEFAULT_TOP_K,
            return_details=True
        )

        result["naive_rag_output"] = (
            rag_result["response"]
        )

        result["retrieved_source"] = (
            rag_result["retrieved_source"]
        )

        result["retrieved_chunk_id"] = (
            rag_result["retrieved_chunk_id"]
        )

        # Retain the top-ranked retrieval distance
        result["retrieval_distance"] = (
            rag_result["retrieved_items"][0]["distance"]
        )

        result["naive_rag_latency_seconds"] = round(
            time.perf_counter() - rag_start,
            3
        )

        result["naive_rag_success"] = bool(
            result["naive_rag_output"]
            and result["naive_rag_output"].strip()
        )

    except Exception as error:
        result["naive_rag_error"] = (
            f"{type(error).__name__}: {error}"
        )
        result["naive_rag_latency_seconds"] = None

    automated_test_results.append(result)

    # Save progress every 10 records
    if (
        len(automated_test_results) % 10 == 0
        or len(automated_test_results) == len(evaluation_df)
    ):
        pd.DataFrame(
            automated_test_results
        ).to_csv(
            CHECKPOINT_PATH,
            index=False
        )

Running Baseline and Naive RAG:   0%|          | 0/50 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info

In [12]:
# Convert collected results into a DataFrame
v1_test_results_df = pd.DataFrame(
    automated_test_results
)

total_duration = (
    time.perf_counter() - evaluation_start_time
)

baseline_success_count = int(
    v1_test_results_df["baseline_success"].sum()
)

rag_success_count = int(
    v1_test_results_df["naive_rag_success"].sum()
)

both_successful_count = int(
    (
        v1_test_results_df["baseline_success"]
        & v1_test_results_df["naive_rag_success"]
    ).sum()
)

# Validate and summarise execution
assert len(v1_test_results_df) == len(evaluation_df)

print("\nAutomated testing completed")
print("=" * 60)
print(f"Evaluation records          : {len(v1_test_results_df):,}")
print(
    f"Successful baseline outputs : "
    f"{baseline_success_count:,}"
)
print(
    f"Successful Naive RAG outputs: "
    f"{rag_success_count:,}"
)
print(
    f"Both systems successful     : "
    f"{both_successful_count:,}"
)
print(
    f"Total execution time        : "
    f"{total_duration / 60:.2f} minutes"
)

if baseline_success_count:
    print(
        f"Average baseline latency    : "
        f"{v1_test_results_df['baseline_latency_seconds'].mean():.2f} seconds"
    )

if rag_success_count:
    print(
        f"Average Naive RAG latency   : "
        f"{v1_test_results_df['naive_rag_latency_seconds'].mean():.2f} seconds"
    )

print(f"\nCheckpoint saved to:\n{CHECKPOINT_PATH}")


# Display representative results
display(
    v1_test_results_df[
        [
            "instruction",
            "true_intent",
            "true_category",
            "baseline_output",
            "retrieved_source",
            "naive_rag_output",
            "baseline_success",
            "naive_rag_success"
        ]
    ].head()
)


Automated testing completed
Evaluation records          : 50
Successful baseline outputs : 50
Successful Naive RAG outputs: 50
Both systems successful     : 50
Total execution time        : 12.82 minutes
Average baseline latency    : 4.44 seconds
Average Naive RAG latency   : 5.97 seconds

Checkpoint saved to:
/content/drive/MyDrive/Hybrid_RAG_Customer_Support/artifacts/notebook_5/automated_test_checkpoint.csv


,instruction,true_intent,true_category,baseline_output,retrieved_source,naive_rag_output,baseline_success,naive_rag_success
0,I try to chat with an operator,contact_human_agent,CONTACT,Hello! I'm here to help you. How can I assist ...,working_hours.md,The live chat and phone support is available f...,True,True
1,can ya direct to me an assistant,contact_human_agent,CONTACT,"I'm sorry, but I can't provide that service di...",working_hours.md,"I'm sorry, but I can't assist with that. Pleas...",True,True
2,I need assistance editing my delivery address,change_shipping_address,SHIPPING,Hello! I'm here to help you with your delivery...,order_tracking.md,Thank you for reaching out. To assist you with...,True,True
3,there is an issue trying to change my shipping...,change_shipping_address,SHIPPING,I'm sorry to hear that you're experiencing iss...,order_tracking.md,I'm sorry to hear you're experiencing issues c...,True,True
4,i dont knoq how i can report errors with payments,payment_issue,PAYMENT,"To report an error on your payment, you should...",account_recovery.md,"To report errors related to your payments, ple...",True,True


#### **3.3.2 Measure Format Adherence [2 marks]**
**The Task:** Validate the syntactic correctness of the generated outputs and report the adherence rate.

**Hints & Tips:**
* For the baseline/RAG free-text responses, "format adherence" means the output is well-formed and non-empty (the strict JSON check applies mainly to the fine-tuned router in NB7).
* Report the percentage of outputs that parsed/validated successfully.

**Learner Inference:** Format adherence tells you how often the system produces usable output before you even check correctness.

In [13]:
# YOUR CODE HERE
def validate_free_text_output(
    output,
    generation_success,
    generation_error
):
    """
    Validate a generated free-text customer-support response.

    A valid output must:
    1. complete without a recorded execution error;
    2. be a string;
    3. contain non-whitespace text;
    4. contain at least one alphabetic character.
    """

    if generation_error is not None and pd.notna(generation_error):
        return False, "Generation error recorded"

    if not bool(generation_success):
        return False, "Generation was unsuccessful"

    if not isinstance(output, str):
        return False, "Output is not a string"

    cleaned_output = output.strip()

    if not cleaned_output:
        return False, "Output is empty"

    if not any(character.isalpha() for character in cleaned_output):
        return False, "Output contains no readable text"

    return True, None

# 1. Validate Baseline outputs
baseline_validation = (
    v1_test_results_df.apply(
        lambda row: validate_free_text_output(
            output=row["baseline_output"],
            generation_success=row["baseline_success"],
            generation_error=row["baseline_error"]
        ),
        axis=1
    )
)

v1_test_results_df["baseline_format_valid"] = [
    result[0]
    for result in baseline_validation
]

v1_test_results_df["baseline_format_error"] = [
    result[1]
    for result in baseline_validation
]

# Validate Naive RAG outputs
rag_validation = (
    v1_test_results_df.apply(
        lambda row: validate_free_text_output(
            output=row["naive_rag_output"],
            generation_success=row["naive_rag_success"],
            generation_error=row["naive_rag_error"]
        ),
        axis=1
    )
)

v1_test_results_df["naive_rag_format_valid"] = [
    result[0]
    for result in rag_validation
]

v1_test_results_df["naive_rag_format_error"] = [
    result[1]
    for result in rag_validation
]

In [14]:
# Calculate adherence rates
total_records = len(v1_test_results_df)

baseline_valid_count = int(
    v1_test_results_df["baseline_format_valid"].sum()
)

rag_valid_count = int(
    v1_test_results_df["naive_rag_format_valid"].sum()
)

baseline_format_adherence = (
    baseline_valid_count / total_records * 100
)

naive_rag_format_adherence = (
    rag_valid_count / total_records * 100
)

format_adherence_summary = pd.DataFrame({
    "system": [
        "Baseline",
        "Naive RAG"
    ],
    "valid_outputs": [
        baseline_valid_count,
        rag_valid_count
    ],
    "invalid_outputs": [
        total_records - baseline_valid_count,
        total_records - rag_valid_count
    ],
    "total_outputs": [
        total_records,
        total_records
    ],
    "format_adherence_percent": [
        baseline_format_adherence,
        naive_rag_format_adherence
    ]
})

# Display results
print("Format-adherence evaluation")
print("=" * 60)
print(f"Evaluation records        : {total_records:,}")
print(
    f"Baseline adherence        : "
    f"{baseline_format_adherence:.2f}%"
)
print(
    f"Naive RAG adherence       : "
    f"{naive_rag_format_adherence:.2f}%"
)

display(
    format_adherence_summary.round({
        "format_adherence_percent": 2
    })
)

# Display invalid cases, if any
invalid_format_rows = v1_test_results_df[
    (~v1_test_results_df["baseline_format_valid"])
    | (~v1_test_results_df["naive_rag_format_valid"])
]

if invalid_format_rows.empty:
    print(
        "\nAll Baseline and Naive RAG outputs were "
        "well-formed and non-empty."
    )
else:
    print("\nInvalid output cases:")

    display(
        invalid_format_rows[
            [
                "instruction",
                "baseline_output",
                "baseline_format_error",
                "naive_rag_output",
                "naive_rag_format_error"
            ]
        ]
    )

Format-adherence evaluation
Evaluation records        : 50
Baseline adherence        : 100.00%
Naive RAG adherence       : 100.00%


,system,valid_outputs,invalid_outputs,total_outputs,format_adherence_percent
0,Baseline,50,0,50,100.0
1,Naive RAG,50,0,50,100.0



All Baseline and Naive RAG outputs were well-formed and non-empty.


### Format Adherence Interpretation

The Baseline system achieved a free-text format-adherence rate of **100%**, while the Naive RAG system achieved **100%**.

For these two systems, format adherence measures whether inference completed successfully and produced a non-empty, readable response. Strict JSON-schema validation is not applicable at this stage because the Baseline and Naive RAG systems generate natural-language customer-support answers rather than structured router outputs.

A high format-adherence rate confirms that the systems execute reliably, but it does not establish that the generated information is factually correct or supported by the SOP documents. Response correctness and grounding are evaluated separately in the following tasks.

#### **3.3.3 Measure Execution Success (ROUGE/BLEU) [2 marks]**
**The Task:** Evaluate semantic similarity of each output against SOP-grounded references using ROUGE-1, ROUGE-L, and BLEU.

**Hints & Tips:**
* Use SOP-grounded references — retrieve the correct SOP per test row so policy-specific language is rewarded.
* Generic references falsely reward vague baseline answers — avoid them.
* `rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)` and `sentence_bleu` with `SmoothingFunction().method1`.

**Learner Inference:** ROUGE/BLEU measure how close the output is to a correct, policy-grounded answer.

In [15]:
# YOUR CODE HERE
%pip install -q rouge-score nltk

  Preparing metadata (setup.py) ... done


In [16]:
import time
import numpy as np

from nltk.translate.bleu_score import (
    SmoothingFunction,
    sentence_bleu
)
from rouge_score import rouge_scorer
from tqdm.auto import tqdm
from IPython.display import display


REFERENCE_TOP_K = 3

REFERENCE_CHECKPOINT_PATH = (
    NOTEBOOK_5_ARTIFACT_DIR
    / "reference_generation_checkpoint.csv"
)

# Retrieve SOP context using ground-truth labels
def retrieve_oracle_context(
    query,
    true_intent,
    true_category,
    k=REFERENCE_TOP_K
):
    """
    Use the ground-truth intent and category to construct a
    policy-specific retrieval query for reference generation.
    """

    oracle_query = (
        f"Category: {str(true_category).strip()}. "
        f"Intent: {str(true_intent).strip()}. "
        f"Customer query: {str(query).strip()}"
    )

    query_embedding = embedding_model.encode(
        [oracle_query],
        normalize_embeddings=True
    ).astype(np.float32)

    results = chroma_collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    if not documents:
        raise RuntimeError(
            "No SOP context was retrieved for reference generation."
        )

    retrieved_items = []

    for rank, (
        document,
        metadata,
        distance
    ) in enumerate(
        zip(documents, metadatas, distances),
        start=1
    ):
        retrieved_items.append({
            "rank": rank,
            "context": document,
            "source_file": metadata.get(
                "source_file",
                "unknown"
            ),
            "chunk_id": metadata.get(
                "chunk_id",
                "unknown"
            ),
            "distance": float(distance)
        })

    return retrieved_items

# Generate a deterministic SOP-grounded reference answer
def generate_reference_answer(
    query,
    true_intent,
    true_category
):
    """
    Generate a concise evaluation reference using oracle
    retrieval based on the ground-truth labels.
    """

    retrieved_items = retrieve_oracle_context(
        query=query,
        true_intent=true_intent,
        true_category=true_category,
        k=REFERENCE_TOP_K
    )

    context_blocks = []

    for item in retrieved_items:
        context_blocks.append(
            f"[Source: {item['source_file']} | "
            f"Chunk: {item['chunk_id']}]\n"
            f"{item['context']}"
        )

    combined_context = "\n\n".join(context_blocks)

    messages = [
        {
            "role": "system",
            "content": (
                "Create a concise reference answer for evaluating "
                "a customer-support system. Use only the supplied "
                "SOP context. Do not invent policies, timelines, "
                "procedures, departments, contact details, refunds, "
                "or guarantees. State that the policy does not provide "
                "enough information when the context is insufficient."
            )
        },
        {
            "role": "user",
            "content": f"""
Customer query:
{str(query).strip()}

Ground-truth category:
{str(true_category).strip()}

Ground-truth intent:
{str(true_intent).strip()}

SOP context:
{combined_context}

Write the policy-grounded reference answer only.
""".strip()
        }
    ]

    reference_output = generate_from_messages(
        messages,
        max_new_tokens=MAX_NEW_TOKENS
    )

    if not reference_output:
        raise RuntimeError(
            "Reference generation returned an empty answer."
        )

    return {
        "reference_output": reference_output,
        "reference_source": retrieved_items[0]["source_file"],
        "reference_chunk_id": retrieved_items[0]["chunk_id"],
        "reference_context": combined_context
    }

In [17]:
# Generate one SOP-grounded reference per evaluation row
reference_results = []
reference_start_time = time.perf_counter()

for row_index, row in tqdm(
    v1_test_results_df.iterrows(),
    total=len(v1_test_results_df),
    desc="Generating SOP-grounded references"
):
    try:
        reference_result = generate_reference_answer(
            query=row["instruction"],
            true_intent=row["true_intent"],
            true_category=row["true_category"]
        )

        reference_results.append({
            "evaluation_id": row["evaluation_id"],
            **reference_result,
            "reference_success": True,
            "reference_error": None
        })

    except Exception as error:
        reference_results.append({
            "evaluation_id": row["evaluation_id"],
            "reference_output": None,
            "reference_source": None,
            "reference_chunk_id": None,
            "reference_context": None,
            "reference_success": False,
            "reference_error": (
                f"{type(error).__name__}: {error}"
            )
        })

    # Save progress periodically
    if (
        len(reference_results) % 10 == 0
        or len(reference_results) == len(v1_test_results_df)
    ):
        pd.DataFrame(reference_results).to_csv(
            REFERENCE_CHECKPOINT_PATH,
            index=False
        )


reference_df = pd.DataFrame(reference_results)

v1_test_results_df = v1_test_results_df.merge(
    reference_df,
    on="evaluation_id",
    how="left",
    validate="one_to_one"
)

reference_success_count = int(
    v1_test_results_df["reference_success"].sum()
)

print("\nReference generation completed")
print("-" * 60)
print(f"Evaluation records       : {len(v1_test_results_df):,}")
print(f"Successful references    : {reference_success_count:,}")
print(
    f"Reference generation time: "
    f"{(time.perf_counter() - reference_start_time) / 60:.2f} minutes"
)

assert reference_success_count == len(v1_test_results_df), (
    "One or more SOP-grounded references could not be generated."
)


Generating SOP-grounded references:   0%|          | 0/50 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info


Reference generation completed
------------------------------------------------------------
Evaluation records       : 50
Successful references    : 50
Reference generation time: 5.85 minutes


In [18]:
# Configure ROUGE and BLEU
rouge_evaluator = rouge_scorer.RougeScorer(
    ["rouge1", "rougeL"],
    use_stemmer=True
)

bleu_smoothing = SmoothingFunction().method1


def calculate_text_metrics(
    reference,
    prediction
):
    """Calculate ROUGE-1, ROUGE-L and sentence BLEU."""

    reference = str(reference).strip()
    prediction = str(prediction).strip()

    if not reference or not prediction:
        return {
            "rouge1": 0.0,
            "rougeL": 0.0,
            "bleu": 0.0
        }

    rouge_scores = rouge_evaluator.score(
        reference,
        prediction
    )

    reference_tokens = reference.lower().split()
    prediction_tokens = prediction.lower().split()

    bleu_score = sentence_bleu(
        [reference_tokens],
        prediction_tokens,
        smoothing_function=bleu_smoothing
    )

    return {
        "rouge1": rouge_scores["rouge1"].fmeasure,
        "rougeL": rouge_scores["rougeL"].fmeasure,
        "bleu": bleu_score
    }

# Score Baseline and Naive RAG outputs
baseline_metric_rows = []
rag_metric_rows = []

for _, row in v1_test_results_df.iterrows():

    baseline_metric_rows.append(
        calculate_text_metrics(
            reference=row["reference_output"],
            prediction=row["baseline_output"]
        )
    )

    rag_metric_rows.append(
        calculate_text_metrics(
            reference=row["reference_output"],
            prediction=row["naive_rag_output"]
        )
    )


v1_test_results_df["baseline_rouge1"] = [
    result["rouge1"]
    for result in baseline_metric_rows
]

v1_test_results_df["baseline_rougeL"] = [
    result["rougeL"]
    for result in baseline_metric_rows
]

v1_test_results_df["baseline_bleu"] = [
    result["bleu"]
    for result in baseline_metric_rows
]

v1_test_results_df["naive_rag_rouge1"] = [
    result["rouge1"]
    for result in rag_metric_rows
]

v1_test_results_df["naive_rag_rougeL"] = [
    result["rougeL"]
    for result in rag_metric_rows
]

v1_test_results_df["naive_rag_bleu"] = [
    result["bleu"]
    for result in rag_metric_rows
]

In [19]:
# Create metric summary
execution_success_summary = pd.DataFrame({
    "system": [
        "Baseline",
        "Naive RAG"
    ],
    "mean_rouge1": [
        v1_test_results_df["baseline_rouge1"].mean(),
        v1_test_results_df["naive_rag_rouge1"].mean()
    ],
    "mean_rougeL": [
        v1_test_results_df["baseline_rougeL"].mean(),
        v1_test_results_df["naive_rag_rougeL"].mean()
    ],
    "mean_bleu": [
        v1_test_results_df["baseline_bleu"].mean(),
        v1_test_results_df["naive_rag_bleu"].mean()
    ]
})

print("\nROUGE and BLEU evaluation")
print("=" * 60)

display(
    execution_success_summary.round(4)
)

# Display representative scored examples
display(
    v1_test_results_df[
        [
            "instruction",
            "true_intent",
            "reference_source",
            "reference_output",
            "baseline_output",
            "baseline_rouge1",
            "baseline_rougeL",
            "baseline_bleu",
            "retrieved_source",
            "naive_rag_output",
            "naive_rag_rouge1",
            "naive_rag_rougeL",
            "naive_rag_bleu"
        ]
    ].head()
)


ROUGE and BLEU evaluation


,system,mean_rouge1,mean_rougeL,mean_bleu
0,Baseline,0.2303,0.1293,0.0118
1,Naive RAG,0.3155,0.1805,0.0334


,instruction,true_intent,reference_source,reference_output,baseline_output,baseline_rouge1,baseline_rougeL,baseline_bleu,retrieved_source,naive_rag_output,naive_rag_rouge1,naive_rag_rougeL,naive_rag_bleu
0,I try to chat with an operator,contact_human_agent,working_hours.md,The policy provides clear guidelines on how lo...,Hello! I'm here to help you. How can I assist ...,0.131387,0.072993,0.000697,working_hours.md,The live chat and phone support is available f...,0.282353,0.117647,0.005821
1,can ya direct to me an assistant,contact_human_agent,working_hours.md,When a customer seeks assistance beyond regula...,"I'm sorry, but I can't provide that service di...",0.251852,0.118519,0.006252,working_hours.md,"I'm sorry, but I can't assist with that. Pleas...",0.252252,0.108108,0.002141
2,I need assistance editing my delivery address,change_shipping_address,order_tracking.md,When a customer needs assistance changing thei...,Hello! I'm here to help you with your delivery...,0.259740,0.103896,0.004689,order_tracking.md,Thank you for reaching out. To assist you with...,0.260355,0.118343,0.005513
3,there is an issue trying to change my shipping...,change_shipping_address,order_tracking.md,### Policy Grounding:\n\nWhen a customer encou...,I'm sorry to hear that you're experiencing iss...,0.131387,0.102190,0.002015,order_tracking.md,I'm sorry to hear you're experiencing issues c...,0.256757,0.175676,0.006436
4,i dont knoq how i can report errors with payments,payment_issue,payment_methods.md,The provided SOP context outlines several impo...,"To report an error on your payment, you should...",0.379630,0.111111,0.003870,account_recovery.md,"To report errors related to your payments, ple...",0.366337,0.158416,0.034819


### ROUGE and BLEU Interpretation

Because the held-out Bitext data does not include human-written SOP-grounded response references, deterministic oracle references were generated using each record’s ground-truth intent and category together with retrieved SOP context.

The Baseline system achieved mean scores of **0.2303** for ROUGE-1, **0.1293** for ROUGE-L, and **0.0118** for BLEU. The Naive RAG system achieved **0.3155**, **0.1805**, and **0.0334**, respectively.

Higher scores indicate that the generated response contains more of the terminology and policy details present in the SOP-grounded reference. However, ROUGE and BLEU primarily measure lexical overlap and do not independently prove factual correctness. These results must therefore be interpreted together with retrieval accuracy and hallucination analysis.

#### **3.3.4 Measure Output Consistency [1 mark]**
**The Task:** Evaluate deterministic behaviour by running the same query multiple times under `do_sample=False` and confirming identical outputs.

**Hints & Tips:**
* Run the same query 3 times; assert all outputs are identical.
* With `do_sample=False, temperature=None`, greedy decoding should be fully deterministic.

**Learner Inference:** Deterministic inference means your evaluation is reproducible — the same input always gives the same output.

In [20]:
# YOUR CODE HERE
CONSISTENCY_RUNS = 3

# Reuse the original ambiguous query from Notebook 3
consistency_query = saved_outputs["test_query"]

print("Consistency test query:")
print(consistency_query)

# Repeat Baseline inference
baseline_repeated_outputs = [
    generate_baseline(consistency_query)
    for _ in range(CONSISTENCY_RUNS)
]

baseline_unique_outputs = set(
    baseline_repeated_outputs
)

baseline_is_consistent = (
    len(baseline_unique_outputs) == 1
)

baseline_consistency_rate = (
    baseline_repeated_outputs.count(
        baseline_repeated_outputs[0]
    )
    / CONSISTENCY_RUNS
    * 100
)

# Repeat Naive RAG inference
rag_repeated_results = [
    generate_naive_rag(
        query=consistency_query,
        k=DEFAULT_TOP_K,
        return_details=True
    )
    for _ in range(CONSISTENCY_RUNS)
]

rag_repeated_outputs = [
    result["response"]
    for result in rag_repeated_results
]

rag_retrieved_sources = [
    result["retrieved_source"]
    for result in rag_repeated_results
]

rag_retrieved_chunks = [
    result["retrieved_chunk_id"]
    for result in rag_repeated_results
]

rag_unique_outputs = set(
    rag_repeated_outputs
)

rag_is_consistent = (
    len(rag_unique_outputs) == 1
)

rag_retrieval_is_consistent = (
    len(set(rag_retrieved_sources)) == 1
    and len(set(rag_retrieved_chunks)) == 1
)

rag_consistency_rate = (
    rag_repeated_outputs.count(
        rag_repeated_outputs[0]
    )
    / CONSISTENCY_RUNS
    * 100
)

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Consistency test query:
My order was supposed to arrive already, but it is still not here. What is happening and when will I receive it?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [21]:
# Validate deterministic behaviour
assert baseline_is_consistent, (
    "Baseline outputs were not identical across repeated runs."
)

assert rag_is_consistent, (
    "Naive RAG outputs were not identical across repeated runs."
)

assert rag_retrieval_is_consistent, (
    "Naive RAG retrieved different chunks across repeated runs."
)

# Summarise results
consistency_summary = pd.DataFrame({
    "system": [
        "Baseline",
        "Naive RAG"
    ],
    "number_of_runs": [
        CONSISTENCY_RUNS,
        CONSISTENCY_RUNS
    ],
    "unique_outputs": [
        len(baseline_unique_outputs),
        len(rag_unique_outputs)
    ],
    "consistency_rate_percent": [
        baseline_consistency_rate,
        rag_consistency_rate
    ],
    "fully_consistent": [
        baseline_is_consistent,
        rag_is_consistent
    ]
})

print("\nOutput-consistency evaluation")
print("=" * 60)

display(
    consistency_summary.round({
        "consistency_rate_percent": 2
    })
)

print(
    "Naive RAG retrieval consistent:",
    rag_retrieval_is_consistent
)

print(
    "Repeated retrieved source:",
    rag_retrieved_sources[0]
)

print(
    "Repeated retrieved chunk:",
    rag_retrieved_chunks[0]
)

# Display repeated outputs
repeated_output_df = pd.DataFrame({
    "run": range(1, CONSISTENCY_RUNS + 1),
    "baseline_output": baseline_repeated_outputs,
    "naive_rag_output": rag_repeated_outputs,
    "retrieved_source": rag_retrieved_sources,
    "retrieved_chunk_id": rag_retrieved_chunks
})

display(repeated_output_df)


Output-consistency evaluation


,system,number_of_runs,unique_outputs,consistency_rate_percent,fully_consistent
0,Baseline,3,1,100.0,True
1,Naive RAG,3,1,100.0,True


Naive RAG retrieval consistent: True
Repeated retrieved source: order_tracking.md
Repeated retrieved chunk: order_tracking_chunk_003


,run,baseline_output,naive_rag_output,retrieved_source,retrieved_chunk_id
0,1,I'm sorry you're experiencing this delay with ...,"Based on our tracking system, your order has b...",order_tracking.md,order_tracking_chunk_003
1,2,I'm sorry you're experiencing this delay with ...,"Based on our tracking system, your order has b...",order_tracking.md,order_tracking_chunk_003
2,3,I'm sorry you're experiencing this delay with ...,"Based on our tracking system, your order has b...",order_tracking.md,order_tracking_chunk_003


### Output Consistency Interpretation

The Baseline and Naive RAG systems were each executed three times using the same customer query and deterministic generation settings.

The Baseline achieved a consistency rate of **100%**, while the Naive RAG system achieved **100%**. The Naive RAG workflow also retrieved the same SOP source and chunk during every run.

This confirms that `do_sample=False` produces reproducible outputs under the current configuration. However, deterministic consistency indicates only that the systems behave repeatedly in the same way; it does not establish that the retrieved policy or generated response is factually correct.

#### **3.3.5 Measure Hallucination Frequency [2 marks]**
**The Task:** Evaluate how often outputs contain unsupported claims, invalid references, missing functionality, or policy violations.

**Hints & Tips:**
* Compare outputs against the retrieved SOP — flag any specific claim (dates, numbers, policies) not supported by the context.
* Report hallucination frequency as a percentage for both Baseline and Naive RAG.

**Learner Inference:** This quantifies the core problem RAG is meant to solve — grounding responses to reduce fabrication.

In [22]:
# YOUR CODE HERE
import re

# Evaluation thresholds
# ------------------------------------------------------------

CLAIM_SIMILARITY_THRESHOLD = 0.30
MISSING_FUNCTIONALITY_THRESHOLD = 0.25

SPECIFIC_CLAIM_TERMS = [
    "refund",
    "replacement",
    "compensation",
    "credit",
    "cancel",
    "escalat",
    "contact",
    "call",
    "email",
    "investigation",
    "eligible",
    "approved",
    "guarantee",
    "delivery",
    "shipping",
    "tracking",
    "password",
    "account",
    "payment",
    "charge",
    "fee",
    "within",
    "business day"
]

ABSOLUTE_COMMITMENT_PATTERNS = [
    r"\bguaranteed?\b",
    r"\bdefinitely\b",
    r"\bautomatically\b",
    r"\bimmediately\b",
    r"\balways\b",
    r"\bwill definitely\b",
    r"\bwill be refunded\b",
    r"\bwill receive\b"
]

NUMERIC_CLAIM_PATTERN = re.compile(
    r"\b\d+(?:\.\d+)?"
    r"(?:\s*(?:-|–|—|to)\s*\d+(?:\.\d+)?)?"
    r"\s*(?:business\s+)?"
    r"(?:minutes?|hours?|days?|weeks?|months?|years?|"
    r"percent|%|dollars?|usd|rupees?|rs\.?)?\b",
    flags=re.IGNORECASE
)

# Shared helper functions
def normalise_text(text):
    """Normalise text for direct claim comparison."""

    return re.sub(
        r"\s+",
        " ",
        str(text)
        .lower()
        .replace("–", "-")
        .replace("—", "-")
    ).strip()


def split_sentences(text):
    """Split generated text into approximate sentences."""

    text = str(text).strip()

    if not text:
        return []

    return [
        sentence.strip()
        for sentence in re.split(
            r"(?<=[.!?])\s+|\n+",
            text
        )
        if sentence.strip()
    ]


def extract_numeric_claims(text):
    """Extract dates, quantities, durations and numeric ranges."""

    return [
        match.group(0).strip()
        for match in NUMERIC_CLAIM_PATTERN.finditer(
            str(text)
        )
    ]


def extract_sources_from_context(context):
    """Extract SOP filenames from formatted context blocks."""

    return set(
        source.strip().lower()
        for source in re.findall(
            r"\[Source:\s*([^|\]]+)",
            str(context),
            flags=re.IGNORECASE
        )
    )


def extract_file_references(text):
    """Extract explicitly mentioned Markdown filenames."""

    return set(
        reference.lower()
        for reference in re.findall(
            r"\b[\w-]+\.md\b",
            str(text),
            flags=re.IGNORECASE
        )
    )


def calculate_semantic_similarity(text_a, text_b):
    """Calculate cosine similarity using normalised MiniLM vectors."""

    if not str(text_a).strip() or not str(text_b).strip():
        return 0.0

    vectors = embedding_model.encode(
        [
            str(text_a).strip(),
            str(text_b).strip()
        ],
        normalize_embeddings=True
    )

    return float(
        np.dot(vectors[0], vectors[1])
    )


def max_context_similarity(sentence, context_segments):
    """Find the strongest semantic support for a sentence."""

    if not sentence or not context_segments:
        return 0.0

    vectors = embedding_model.encode(
        [sentence] + context_segments,
        normalize_embeddings=True
    )

    sentence_vector = vectors[0]
    context_vectors = vectors[1:]

    similarities = context_vectors @ sentence_vector

    return float(similarities.max())


def audit_generated_output(
    output,
    support_context,
    reference_output,
    allowed_sources
):
    """
    Audit one generated answer for unsupported or missing content.
    """

    output = "" if pd.isna(output) else str(output).strip()
    support_context = (
        ""
        if pd.isna(support_context)
        else str(support_context).strip()
    )
    reference_output = (
        ""
        if pd.isna(reference_output)
        else str(reference_output).strip()
    )

    if not output:
        return {
            "hallucination_detected": True,
            "functionally_correct": False,
            "missing_functionality": True,
            "reference_similarity": 0.0,
            "unsupported_numeric_claims": "",
            "unsupported_claim_sentences": "",
            "invalid_references": "",
            "policy_violation": False,
            "hallucination_reasons": "Empty or missing response"
        }

    normalised_context = normalise_text(
        support_context
    )

    context_segments = [
        segment.strip()
        for segment in re.split(
            r"\n\s*\n",
            support_context
        )
        if segment.strip()
    ]

    output_sentences = split_sentences(output)

    # Check unsupported numeric and timeline claims
    numeric_claims = extract_numeric_claims(output)

    unsupported_numeric_claims = [
        claim
        for claim in numeric_claims
        if normalise_text(claim) not in normalised_context
    ]

    # Check policy-sensitive sentences for semantic support
    candidate_claim_sentences = []

    for sentence in output_sentences:
        normalised_sentence = normalise_text(sentence)

        contains_policy_term = any(
            term in normalised_sentence
            for term in SPECIFIC_CLAIM_TERMS
        )

        contains_number = bool(
            extract_numeric_claims(sentence)
        )

        if contains_policy_term or contains_number:
            candidate_claim_sentences.append(sentence)

    unsupported_claim_sentences = []

    for sentence in candidate_claim_sentences:
        similarity = max_context_similarity(
            sentence,
            context_segments
        )

        if similarity < CLAIM_SIMILARITY_THRESHOLD:
            unsupported_claim_sentences.append(
                f"{sentence} [support={similarity:.3f}]"
            )

    # Check explicit SOP file references
    mentioned_references = extract_file_references(
        output
    )

    allowed_sources = {
        str(source).strip().lower()
        for source in allowed_sources
    }

    invalid_references = sorted(
        mentioned_references - allowed_sources
    )

    # Check unsupported absolute promises
    policy_violation_phrases = []

    for pattern in ABSOLUTE_COMMITMENT_PATTERNS:
        matches = re.findall(
            pattern,
            output,
            flags=re.IGNORECASE
        )

        for match in matches:
            phrase = (
                match
                if isinstance(match, str)
                else " ".join(match)
            )

            if normalise_text(phrase) not in normalised_context:
                policy_violation_phrases.append(
                    normalise_text(phrase)
                )

    policy_violation = bool(
        policy_violation_phrases
    )

    # Check missing functionality against oracle reference
    reference_similarity = (
        calculate_semantic_similarity(
            output,
            reference_output
        )
    )

    missing_functionality = (
        reference_similarity
        < MISSING_FUNCTIONALITY_THRESHOLD
    )

    # Final decision
    hallucination_reasons = []

    if unsupported_numeric_claims:
        hallucination_reasons.append(
            "Unsupported numeric or timeline claim"
        )

    if unsupported_claim_sentences:
        hallucination_reasons.append(
            "Policy claim weakly supported by context"
        )

    if invalid_references:
        hallucination_reasons.append(
            "Invalid SOP reference"
        )

    if policy_violation:
        hallucination_reasons.append(
            "Unsupported absolute promise"
        )

    if missing_functionality:
        hallucination_reasons.append(
            "Missing required policy functionality"
        )

    hallucination_detected = bool(
        hallucination_reasons
    )

    functionally_correct = (
        not hallucination_detected
        and not missing_functionality
    )

    return {
        "hallucination_detected": hallucination_detected,
        "functionally_correct": functionally_correct,
        "missing_functionality": missing_functionality,
        "reference_similarity": reference_similarity,
        "unsupported_numeric_claims": " || ".join(
            unsupported_numeric_claims
        ),
        "unsupported_claim_sentences": " || ".join(
            unsupported_claim_sentences
        ),
        "invalid_references": " || ".join(
            invalid_references
        ),
        "policy_violation": policy_violation,
        "hallucination_reasons": " || ".join(
            hallucination_reasons
        )
    }

In [23]:
# Audit every evaluation record
baseline_audit_rows = []
naive_rag_audit_rows = []

for _, row in tqdm(
    v1_test_results_df.iterrows(),
    total=len(v1_test_results_df),
    desc="Auditing hallucinations"
):
    # Baseline has no retrieved context, so compare it with the
    # oracle SOP evidence prepared in Task 3.3.3.
    baseline_context = row["reference_context"]

    baseline_sources = extract_sources_from_context(
        baseline_context
    )

    baseline_audit_rows.append(
        audit_generated_output(
            output=row["baseline_output"],
            support_context=baseline_context,
            reference_output=row["reference_output"],
            allowed_sources=baseline_sources
        )
    )

    # Recreate the actual raw-query context used by Naive RAG.
    retrieved_items = retrieve_naive_context(
        query=row["instruction"],
        k=DEFAULT_TOP_K
    )

    naive_context = "\n\n".join(
        f"[Source: {item['source_file']} | "
        f"Chunk: {item['chunk_id']}]\n"
        f"{item['context']}"
        for item in retrieved_items
    )

    naive_sources = {
        item["source_file"]
        for item in retrieved_items
    }

    naive_rag_audit_rows.append(
        audit_generated_output(
            output=row["naive_rag_output"],
            support_context=naive_context,
            reference_output=row["reference_output"],
            allowed_sources=naive_sources
        )
    )

# Add audit results to the evaluation DataFrame
baseline_audit_df = pd.DataFrame(
    baseline_audit_rows
).add_prefix("baseline_")

naive_rag_audit_df = pd.DataFrame(
    naive_rag_audit_rows
).add_prefix("naive_rag_")

v1_test_results_df = pd.concat(
    [
        v1_test_results_df.reset_index(drop=True),
        baseline_audit_df.reset_index(drop=True),
        naive_rag_audit_df.reset_index(drop=True)
    ],
    axis=1
)

# Calculate aggregate hallucination frequencies
total_evaluated = len(v1_test_results_df)

baseline_hallucination_count = int(
    v1_test_results_df[
        "baseline_hallucination_detected"
    ].sum()
)

naive_rag_hallucination_count = int(
    v1_test_results_df[
        "naive_rag_hallucination_detected"
    ].sum()
)

baseline_hallucination_frequency = (
    baseline_hallucination_count
    / total_evaluated
    * 100
)

naive_rag_hallucination_frequency = (
    naive_rag_hallucination_count
    / total_evaluated
    * 100
)

baseline_functional_correctness = (
    v1_test_results_df[
        "baseline_functionally_correct"
    ].mean()
    * 100
)

naive_rag_functional_correctness = (
    v1_test_results_df[
        "naive_rag_functionally_correct"
    ].mean()
    * 100
)


hallucination_summary = pd.DataFrame({
    "system": [
        "Baseline",
        "Naive RAG"
    ],
    "hallucinated_outputs": [
        baseline_hallucination_count,
        naive_rag_hallucination_count
    ],
    "total_outputs": [
        total_evaluated,
        total_evaluated
    ],
    "hallucination_frequency_percent": [
        baseline_hallucination_frequency,
        naive_rag_hallucination_frequency
    ],
    "functional_correctness_percent": [
        baseline_functional_correctness,
        naive_rag_functional_correctness
    ]
})

Auditing hallucinations:   0%|          | 0/50 [00:00<?, ?it/s]

In [24]:
# Display results
print("Hallucination-frequency evaluation")
print("=" * 65)
print(f"Evaluation records             : {total_evaluated}")
print(
    f"Baseline hallucination rate    : "
    f"{baseline_hallucination_frequency:.2f}%"
)
print(
    f"Naive RAG hallucination rate   : "
    f"{naive_rag_hallucination_frequency:.2f}%"
)
print(
    f"Baseline functional correctness: "
    f"{baseline_functional_correctness:.2f}%"
)
print(
    f"Naive RAG functional correctness: "
    f"{naive_rag_functional_correctness:.2f}%"
)

display(
    hallucination_summary.round(2)
)

# Display representative flagged cases
flagged_examples = v1_test_results_df[
    v1_test_results_df[
        "baseline_hallucination_detected"
    ]
    | v1_test_results_df[
        "naive_rag_hallucination_detected"
    ]
]

print("\nRepresentative flagged cases:")

display(
    flagged_examples[
        [
            "instruction",
            "reference_source",
            "baseline_output",
            "baseline_hallucination_reasons",
            "retrieved_source",
            "naive_rag_output",
            "naive_rag_hallucination_reasons"
        ]
    ].head(10)
)

Hallucination-frequency evaluation
Evaluation records             : 50
Baseline hallucination rate    : 52.00%
Naive RAG hallucination rate   : 76.00%
Baseline functional correctness: 48.00%
Naive RAG functional correctness: 24.00%


,system,hallucinated_outputs,total_outputs,hallucination_frequency_percent,functional_correctness_percent
0,Baseline,26,50,52.0,48.0
1,Naive RAG,38,50,76.0,24.0



Representative flagged cases:


,instruction,reference_source,baseline_output,baseline_hallucination_reasons,retrieved_source,naive_rag_output,naive_rag_hallucination_reasons
0,I try to chat with an operator,working_hours.md,Hello! I'm here to help you. How can I assist ...,Policy claim weakly supported by context || Mi...,working_hours.md,The live chat and phone support is available f...,Unsupported numeric or timeline claim
4,i dont knoq how i can report errors with payments,payment_methods.md,"To report an error on your payment, you should...",,account_recovery.md,"To report errors related to your payments, ple...",Unsupported numeric or timeline claim || Polic...
5,need help to check if there are any news on th...,refund_policy.md,"I'm sorry, but I am not able to provide real-t...",Missing required policy functionality,order_tracking.md,We apologize for any inconvenience. To assist ...,Policy claim weakly supported by context || Un...
6,could you help me to acquire something?,order_tracking.md,I'm here to assist with any questions or conce...,Policy claim weakly supported by context || Mi...,technical_troubleshooting.md,"I'm sorry, but I cannot assist with acquiring ...",
7,assistance canceling the last order,subscription_cancellation.md,I'm sorry to hear that you're having trouble w...,,subscription_cancellation.md,"Dear [Customer],\n\nThank you for reaching out...",Unsupported numeric or timeline claim || Polic...
8,I want to inform of problems with signup,technical_troubleshooting.md,Thank you for reaching out. I'm here to help i...,,password_reset.md,"Dear [Customer's Name],\n\nThank you for reach...",Unsupported numeric or timeline claim || Polic...
9,can you help me sending my feedback for your s...,data_privacy.md,Of course! I'd be happy to assist you in provi...,Policy claim weakly supported by context,working_hours.md,"I'm sorry, but I can't assist with that. Pleas...",
10,I do not know how I can see the withdrawal fee,subscription_cancellation.md,"To view your withdrawal fee, you should log in...",Policy claim weakly supported by context,payment_methods.md,The withdrawal fee is not visible as it is par...,Policy claim weakly supported by context
11,where could i check order {{Order Number}} fuc...,order_tracking.md,To check the current status of your order with...,Unsupported numeric or timeline claim || Polic...,order_tracking.md,To check the current status of your order with...,Unsupported numeric or timeline claim || Polic...
12,I'm trying to notify of errors with online pay...,payment_methods.md,"To help you better, could you please provide m...",Policy claim weakly supported by context,technical_troubleshooting.md,To address your concern about notifying errors...,Unsupported numeric or timeline claim || Polic...


### Hallucination Frequency Interpretation

The Baseline system produced unsupported, incomplete, or policy-inconsistent responses in **26 of 50 cases**, corresponding to a hallucination frequency of **52%**.

The Naive RAG system produced flagged responses in **38 of 50 cases**, corresponding to a hallucination frequency of **76%**.

The audit compared Baseline responses with oracle SOP evidence and Naive RAG responses with the context actually retrieved from the raw query. It considered unsupported numeric claims, weakly grounded policy statements, invalid SOP references, unsupported promises, and failure to provide the required policy functionality.

A lower Naive RAG hallucination frequency would indicate that retrieval improved grounding. However, cases where Naive RAG retrieved a semantically related but incorrect SOP can still produce incomplete or policy-inappropriate responses.

### **Task 3.4: Analyse Retrieval Impact**

#### **3.4.1 Compare Baseline and Solution V1 [4 marks]**
**The Task:** Quantify the impact of retrieval by comparing aggregate scores across Functional Correctness, Consistency, and Hallucination Frequency, with percentage changes.

**Hints & Tips:**
* Build a summary table: Baseline vs Naive RAG for each metric.
* Compute improvement percentages: `(rag - base) / base * 100`.
* Document WHERE retrieval helps and where it doesn't — both motivate Stage 4.

**Learner Inference:** This isolates retrieval's independent contribution before fine-tuning enters the picture.

In [25]:
# YOUR CODE HERE
# Helper for relative improvement
def calculate_relative_improvement(
    baseline_value,
    rag_value,
    higher_is_better=True
):
    """
    Calculate relative improvement.

    For higher-is-better metrics:
        (RAG - Baseline) / Baseline * 100

    For lower-is-better metrics:
        (Baseline - RAG) / Baseline * 100
    """

    if baseline_value == 0:
        return np.nan

    if higher_is_better:
        return (
            (rag_value - baseline_value)
            / baseline_value
            * 100
        )

    return (
        (baseline_value - rag_value)
        / baseline_value
        * 100
    )

# Collect aggregate metric values
baseline_metrics = {
    "Functional Correctness": (
        baseline_functional_correctness
    ),
    "Output Consistency": (
        baseline_consistency_rate
    ),
    "Hallucination Frequency": (
        baseline_hallucination_frequency
    )
}

rag_metrics = {
    "Functional Correctness": (
        naive_rag_functional_correctness
    ),
    "Output Consistency": (
        rag_consistency_rate
    ),
    "Hallucination Frequency": (
        naive_rag_hallucination_frequency
    )
}

# Build the primary comparison table
comparison_rows = []

for metric_name in baseline_metrics:

    baseline_value = baseline_metrics[metric_name]
    rag_value = rag_metrics[metric_name]

    higher_is_better = (
        metric_name != "Hallucination Frequency"
    )

    if higher_is_better:
        beneficial_change = (
            rag_value - baseline_value
        )
    else:
        beneficial_change = (
            baseline_value - rag_value
        )

    relative_improvement = (
        calculate_relative_improvement(
            baseline_value=baseline_value,
            rag_value=rag_value,
            higher_is_better=higher_is_better
        )
    )

    comparison_rows.append({
        "metric": metric_name,
        "preferred_direction": (
            "Higher"
            if higher_is_better
            else "Lower"
        ),
        "baseline_percent": baseline_value,
        "naive_rag_percent": rag_value,
        "beneficial_change_percentage_points": (
            beneficial_change
        ),
        "relative_improvement_percent": (
            relative_improvement
        ),
        "retrieval_effect": (
            "Improved"
            if beneficial_change > 0
            else (
                "Unchanged"
                if beneficial_change == 0
                else "Declined"
            )
        )
    })


retrieval_impact_summary = pd.DataFrame(
    comparison_rows
)


print("Baseline versus Solution V1")
print("=" * 70)

display(
    retrieval_impact_summary.round({
        "baseline_percent": 2,
        "naive_rag_percent": 2,
        "beneficial_change_percentage_points": 2,
        "relative_improvement_percent": 2
    })
)

Baseline versus Solution V1


,metric,preferred_direction,baseline_percent,naive_rag_percent,beneficial_change_percentage_points,relative_improvement_percent,retrieval_effect
0,Functional Correctness,Higher,48.0,24.0,-24.0,-50.00,Declined
1,Output Consistency,Higher,100.0,100.0,0.0,0.00,Unchanged
2,Hallucination Frequency,Lower,52.0,76.0,-24.0,-46.15,Declined


In [26]:
# Add supporting ROUGE, BLEU and format metrics
supplementary_metric_summary = pd.DataFrame({
    "metric": [
        "Format Adherence",
        "ROUGE-1",
        "ROUGE-L",
        "BLEU"
    ],
    "baseline": [
        baseline_format_adherence,
        v1_test_results_df[
            "baseline_rouge1"
        ].mean(),
        v1_test_results_df[
            "baseline_rougeL"
        ].mean(),
        v1_test_results_df[
            "baseline_bleu"
        ].mean()
    ],
    "naive_rag": [
        naive_rag_format_adherence,
        v1_test_results_df[
            "naive_rag_rouge1"
        ].mean(),
        v1_test_results_df[
            "naive_rag_rougeL"
        ].mean(),
        v1_test_results_df[
            "naive_rag_bleu"
        ].mean()
    ]
})

supplementary_metric_summary[
    "absolute_change"
] = (
    supplementary_metric_summary["naive_rag"]
    - supplementary_metric_summary["baseline"]
)

supplementary_metric_summary[
    "relative_change_percent"
] = supplementary_metric_summary.apply(
    lambda row: (
        np.nan
        if row["baseline"] == 0
        else (
            (row["naive_rag"] - row["baseline"])
            / row["baseline"]
            * 100
        )
    ),
    axis=1
)

print("\nSupporting quality metrics")
print("=" * 70)

display(
    supplementary_metric_summary.round(4)
)

# Identify where retrieval helped and where it did not
baseline_correct = (
    v1_test_results_df[
        "baseline_functionally_correct"
    ].astype(bool)
)

rag_correct = (
    v1_test_results_df[
        "naive_rag_functionally_correct"
    ].astype(bool)
)

v1_test_results_df["retrieval_impact_case"] = np.select(
    [
        (~baseline_correct) & rag_correct,
        baseline_correct & (~rag_correct),
        baseline_correct & rag_correct
    ],
    [
        "Retrieval helped",
        "Retrieval hurt",
        "Both correct"
    ],
    default="Both incorrect"
)

retrieval_case_summary = (
    v1_test_results_df[
        "retrieval_impact_case"
    ]
    .value_counts()
    .reindex(
        [
            "Retrieval helped",
            "Retrieval hurt",
            "Both correct",
            "Both incorrect"
        ],
        fill_value=0
    )
    .rename_axis("case_type")
    .reset_index(name="record_count")
)

retrieval_case_summary["percentage"] = (
    retrieval_case_summary["record_count"]
    / len(v1_test_results_df)
    * 100
)

print("\nPer-record retrieval impact")
print("=" * 70)

display(
    retrieval_case_summary.round({
        "percentage": 2
    })
)


Supporting quality metrics


,metric,baseline,naive_rag,absolute_change,relative_change_percent
0,Format Adherence,100.0000,100.0000,0.0000,0.0000
1,ROUGE-1,0.2303,0.3155,0.0851,36.9698
2,ROUGE-L,0.1293,0.1805,0.0511,39.5164
3,BLEU,0.0118,0.0334,0.0215,182.1268



Per-record retrieval impact


,case_type,record_count,percentage
0,Retrieval helped,4,8.0
1,Retrieval hurt,16,32.0
2,Both correct,8,16.0
3,Both incorrect,22,44.0


In [27]:
# Compare raw-query source with oracle source
v1_test_results_df["retrieval_source_match"] = (
    v1_test_results_df["retrieved_source"]
    .fillna("")
    .str.lower()
    .str.strip()
    ==
    v1_test_results_df["reference_source"]
    .fillna("")
    .str.lower()
    .str.strip()
)

retrieval_source_match_rate = (
    v1_test_results_df[
        "retrieval_source_match"
    ].mean()
    * 100
)

print(
    f"\nTop-1 source agreement with oracle retrieval: "
    f"{retrieval_source_match_rate:.2f}%"
)

# Show examples where retrieval helped or failed
v1_test_results_df[
    "reference_similarity_change"
] = (
    v1_test_results_df[
        "naive_rag_reference_similarity"
    ]
    - v1_test_results_df[
        "baseline_reference_similarity"
    ]
)

print("\nExamples where retrieval helped:")

helped_examples = (
    v1_test_results_df[
        v1_test_results_df[
            "retrieval_impact_case"
        ] == "Retrieval helped"
    ]
    .sort_values(
        "reference_similarity_change",
        ascending=False
    )
)

if helped_examples.empty:
    print("No cases changed from incorrect to correct.")
else:
    display(
        helped_examples[
            [
                "instruction",
                "reference_source",
                "retrieved_source",
                "baseline_output",
                "naive_rag_output",
                "reference_similarity_change"
            ]
        ].head(5)
    )


print("\nExamples where retrieval did not help:")

failed_examples = (
    v1_test_results_df[
        v1_test_results_df[
            "retrieval_impact_case"
        ].isin([
            "Retrieval hurt",
            "Both incorrect"
        ])
    ]
    .sort_values(
        "reference_similarity_change",
        ascending=True
    )
)

if failed_examples.empty:
    print("No failed retrieval-impact cases were found.")
else:
    display(
        failed_examples[
            [
                "instruction",
                "reference_source",
                "retrieved_source",
                "baseline_output",
                "naive_rag_output",
                "naive_rag_hallucination_reasons",
                "reference_similarity_change"
            ]
        ].head(5)
    )


Top-1 source agreement with oracle retrieval: 62.00%

Examples where retrieval helped:


,instruction,reference_source,retrieved_source,baseline_output,naive_rag_output,reference_similarity_change
35,I need assistance to get a rebate of my money,refund_policy.md,refund_policy.md,"I'm sorry, but I am unable to assist with that...",The customer can request a refund if they purc...,0.428909
6,could you help me to acquire something?,order_tracking.md,technical_troubleshooting.md,I'm here to assist with any questions or conce...,"I'm sorry, but I cannot assist with acquiring ...",0.237313
49,can you help me to set a shipping address up?,order_tracking.md,order_tracking.md,"Sure, I can help with that! To set up your shi...","I'm sorry, but I currently don't have access t...",0.066075
9,can you help me sending my feedback for your s...,data_privacy.md,working_hours.md,Of course! I'd be happy to assist you in provi...,"I'm sorry, but I can't assist with that. Pleas...",-0.367799



Examples where retrieval did not help:


,instruction,reference_source,retrieved_source,baseline_output,naive_rag_output,naive_rag_hallucination_reasons,reference_similarity_change
46,know if there is anything new on my rebate,billing_disputes.md,refund_policy.md,"I'm sorry, but as an AI language model, I don'...","According to our refund policy, you can reques...",Missing required policy functionality,-0.192074
31,I don't know how to lodge a condumer reclamation,product_return.md,shipping_delays.md,"To file a consumer complaint, you can visit yo...","Dear valued customer,\n\nThank you for reachin...",Unsupported numeric or timeline claim || Polic...,-0.126728
22,want help filing a cudtomer claim against ur o...,product_return.md,product_return.md,I'm sorry to hear that you're having trouble w...,"I'm sorry, but I cannot assist with this reque...",Missing required policy functionality,-0.123396
43,"I need to speak with customer support, help me",working_hours.md,technical_troubleshooting.md,"Sure, I can assist you in speaking with custom...",Thank you for reaching out. I'm here to assist...,Missing required policy functionality,-0.085456
23,need help checking how long it takes for the i...,order_tracking.md,order_tracking.md,"I'm sorry, but as an AI language model, I don'...",To determine when your item is expected to arr...,Policy claim weakly supported by context,-0.070333


### Retrieval Impact Interpretation

The Naive RAG system achieved a functional-correctness rate of **24%**, compared with **48%** for the Baseline system. This represents a decline of **24 percentage points**, equivalent to a **50% relative reduction** in functional correctness.

Both systems achieved an output-consistency rate of **100%**, confirming that retrieval did not affect reproducibility under deterministic generation.

The Baseline hallucination frequency was **52%**, while the Naive RAG hallucination frequency was **76%**. Therefore, hallucination frequency increased by **24 percentage points**, representing a **46.15% relative increase** compared with the Baseline.

At the individual-record level, retrieval improved the outcome in **4 cases (8%)**, reduced performance in **16 cases (32%)**, and left both systems incorrect in **22 cases (44%)**. Both systems were correct in the remaining **8 cases (16%)**. The raw-query retriever selected the same top-ranked SOP as the oracle retrieval process in **62%** of evaluated cases.

Naive RAG nevertheless improved lexical similarity, with relative increases of approximately **36.97% in ROUGE-1**, **39.52% in ROUGE-L**, and **182.13% in BLEU**. This indicates that retrieved context introduced more policy-related language, but higher lexical overlap did not consistently translate into better functional correctness or lower hallucination.

Overall, retrieval helped when the raw customer query identified the correct SOP. However, ambiguous or noisy queries frequently retrieved a related but inappropriate policy, causing the generated response to become incomplete or policy-inconsistent. This result motivates the fine-tuned intent router in Solution V2, which will use structured intent and category information to improve retrieval precision.

---
## Save Artifacts

In [28]:
# YOUR CODE HERE
# Define the output location
NOTEBOOK_5_ARTIFACT_DIR = (
    DRIVE_PROJECT_DIR
    / "artifacts"
    / "notebook_5"
)

V1_METRICS_PATH = (
    NOTEBOOK_5_ARTIFACT_DIR
    / "v1_metrics.csv"
)

NOTEBOOK_5_ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Add aggregate consistency results to each record
v1_test_results_df[
    "baseline_consistency_rate_percent"
] = baseline_consistency_rate

v1_test_results_df[
    "naive_rag_consistency_rate_percent"
] = rag_consistency_rate

# Select the evidence required for later comparison
V1_EXPORT_COLUMNS = [
    # Test record
    "evaluation_id",
    "instruction",
    "true_intent",
    "true_category",

    # Generated outputs
    "baseline_output",
    "naive_rag_output",

    # Retrieval evidence
    "retrieved_source",
    "retrieved_chunk_id",
    "retrieval_distance",
    "reference_source",
    "retrieval_source_match",

    # Execution and format adherence
    "baseline_success",
    "naive_rag_success",
    "baseline_format_valid",
    "naive_rag_format_valid",
    "baseline_latency_seconds",
    "naive_rag_latency_seconds",

    # ROUGE and BLEU
    "baseline_rouge1",
    "baseline_rougeL",
    "baseline_bleu",
    "naive_rag_rouge1",
    "naive_rag_rougeL",
    "naive_rag_bleu",

    # Functional correctness and hallucination
    "baseline_functionally_correct",
    "naive_rag_functionally_correct",
    "baseline_hallucination_detected",
    "naive_rag_hallucination_detected",
    "baseline_missing_functionality",
    "naive_rag_missing_functionality",
    "baseline_reference_similarity",
    "naive_rag_reference_similarity",
    "baseline_hallucination_reasons",
    "naive_rag_hallucination_reasons",

    # Retrieval impact
    "retrieval_impact_case",
    "reference_similarity_change",

    # Consistency
    "baseline_consistency_rate_percent",
    "naive_rag_consistency_rate_percent"
]

missing_export_columns = [
    column
    for column in V1_EXPORT_COLUMNS
    if column not in v1_test_results_df.columns
]

if missing_export_columns:
    raise ValueError(
        "The following required evaluation columns are missing:\n"
        f"{missing_export_columns}"
    )

v1_metrics_df = (
    v1_test_results_df[V1_EXPORT_COLUMNS]
    .copy()
    .reset_index(drop=True)
)

In [29]:
# Validate the final artefact
assert len(v1_metrics_df) == len(evaluation_df)
assert v1_metrics_df["evaluation_id"].is_unique
assert not v1_metrics_df[
    ["instruction", "baseline_output", "naive_rag_output"]
].isna().any().any()

# Save atomically
TEMP_V1_METRICS_PATH = (
    V1_METRICS_PATH.with_suffix(".tmp")
)

v1_metrics_df.to_csv(
    TEMP_V1_METRICS_PATH,
    index=False
)

os.replace(
    TEMP_V1_METRICS_PATH,
    V1_METRICS_PATH
)

# Reload and confirm
saved_v1_metrics = pd.read_csv(
    V1_METRICS_PATH
)

assert len(saved_v1_metrics) == len(v1_metrics_df)
assert saved_v1_metrics.columns.tolist() == (
    v1_metrics_df.columns.tolist()
)

print("Notebook 5 artefact saved successfully")
print("-" * 60)
print(f"Output path       : {V1_METRICS_PATH}")
print(f"Evaluation records: {len(saved_v1_metrics):,}")
print(f"Saved columns     : {len(saved_v1_metrics.columns)}")

print("\nAggregate results retained in the artefact:")
print(
    f"Baseline functional correctness : "
    f"{baseline_functional_correctness:.2f}%"
)
print(
    f"Naive RAG functional correctness: "
    f"{naive_rag_functional_correctness:.2f}%"
)
print(
    f"Baseline hallucination frequency: "
    f"{baseline_hallucination_frequency:.2f}%"
)
print(
    f"Naive RAG hallucination frequency: "
    f"{naive_rag_hallucination_frequency:.2f}%"
)
print(
    f"Baseline consistency            : "
    f"{baseline_consistency_rate:.2f}%"
)
print(
    f"Naive RAG consistency           : "
    f"{rag_consistency_rate:.2f}%"
)

display(saved_v1_metrics.head())

Notebook 5 artefact saved successfully
------------------------------------------------------------
Output path       : /content/drive/MyDrive/Hybrid_RAG_Customer_Support/artifacts/notebook_5/v1_metrics.csv
Evaluation records: 50
Saved columns     : 37

Aggregate results retained in the artefact:
Baseline functional correctness : 48.00%
Naive RAG functional correctness: 24.00%
Baseline hallucination frequency: 52.00%
Naive RAG hallucination frequency: 76.00%
Baseline consistency            : 100.00%
Naive RAG consistency           : 100.00%


,evaluation_id,instruction,true_intent,true_category,baseline_output,naive_rag_output,retrieved_source,retrieved_chunk_id,retrieval_distance,reference_source,...,baseline_missing_functionality,naive_rag_missing_functionality,baseline_reference_similarity,naive_rag_reference_similarity,baseline_hallucination_reasons,naive_rag_hallucination_reasons,retrieval_impact_case,reference_similarity_change,baseline_consistency_rate_percent,naive_rag_consistency_rate_percent
0,0,I try to chat with an operator,contact_human_agent,CONTACT,Hello! I'm here to help you. How can I assist ...,The live chat and phone support is available f...,working_hours.md,working_hours_chunk_000,0.625501,working_hours.md,...,True,False,0.100527,0.541641,Policy claim weakly supported by context || Mi...,Unsupported numeric or timeline claim,Both incorrect,0.441114,100.0,100.0
1,1,can ya direct to me an assistant,contact_human_agent,CONTACT,"I'm sorry, but I can't provide that service di...","I'm sorry, but I can't assist with that. Pleas...",working_hours.md,working_hours_chunk_004,0.668749,working_hours.md,...,False,False,0.573780,0.482082,NaN,NaN,Both correct,-0.091698,100.0,100.0
2,2,I need assistance editing my delivery address,change_shipping_address,SHIPPING,Hello! I'm here to help you with your delivery...,Thank you for reaching out. To assist you with...,order_tracking.md,order_tracking_chunk_004,0.499231,order_tracking.md,...,False,False,0.657872,0.661969,NaN,NaN,Both correct,0.004096,100.0,100.0
3,3,there is an issue trying to change my shipping...,change_shipping_address,SHIPPING,I'm sorry to hear that you're experiencing iss...,I'm sorry to hear you're experiencing issues c...,order_tracking.md,order_tracking_chunk_004,0.591998,order_tracking.md,...,False,False,0.648713,0.647993,NaN,NaN,Both correct,-0.000720,100.0,100.0
4,4,i dont knoq how i can report errors with payments,payment_issue,PAYMENT,"To report an error on your payment, you should...","To report errors related to your payments, ple...",account_recovery.md,account_recovery_chunk_001,0.413222,payment_methods.md,...,False,False,0.567591,0.664150,NaN,Unsupported numeric or timeline claim || Polic...,Retrieval hurt,0.096559,100.0,100.0


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 6.**

- [x] ChromaDB reloaded from `./chroma_db/`
- [x] **3.3.1** Automated testing run across full test set
- [x] **3.3.2** Format adherence measured
- [x] **3.3.3** ROUGE/BLEU computed with SOP-grounded references
- [x] **3.3.4** Output consistency (determinism) verified
- [x] **3.3.5** Hallucination frequency quantified
- [x] **3.4.1** Retrieval impact quantified with improvement %
- [x] **`v1_metrics.csv` saved** ← _Evidence for comparative analysis_

**If any item is unchecked, fix it before moving on.**